# Модуль 03 · Урок 4. Аудит качества и воспроизводимая очистка

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Данные сначала проверяют, потом анализируют
Аналитический вывод нельзя считать надёжным, если неизвестно, сколько строк было отброшено, есть ли дубликаты и существуют ли недопустимые значения. Начните с профиля: число строк, пропуски, уникальные ключи, минимумы и максимумы, доля корректных значений. Аудит — это **измерение**, очистка — отдельное **изменение**. Не смешивайте эти действия в одной длинной функции.

In [ ]:
import pandas as pd
sales = pd.DataFrame({'id':[1,2,2,3],'amount':[100,-20,30,None]})
print('строк:',len(sales),'дубли ID:',sales.id.duplicated().sum())
print('отрицательных сумм:',sales.amount.lt(0).sum())

## 2. Правила предметной области
У каждой колонки есть **технический тип** и **бизнес-ограничения**. Для заказа: количество должно быть целым неотрицательным, цена — неотрицательной, статус — из справочника. Для событий: дата должна разбираться, пользователь — быть идентифицированным. Лучше явно записать ограничения в документации и превратить их в проверяемые условия. Не пытайтесь «починить» неизвестный статус случайной заменой: это может исказить отчёт.

## 3. Разделение валидных и проблемных строк
Удобный результат проверки — логическая маска `valid_mask`. `df.loc[valid_mask]` даёт корректные строки, `df.loc[~valid_mask]` — очередь на исправление. Таким образом, проблемные записи не пропадают бесследно. Функции, возвращающие новые объекты вместо изменения исходника, проще тестировать и повторно применять.

In [ ]:
payments = pd.DataFrame({'amount':[0,50,-4,None],'status':['ok','ok','ok','unknown']})
valid = payments.amount.ge(0) & payments.status.eq('ok')
print(payments.loc[valid]); print(payments.loc[~valid])

## 4. Идемпотентность и проверки после очистки
Идемпотентная очистка означает, что повторное применение функции не приводит к дальнейшим изменениям (`clean(clean(df)) == clean(df)`). Например, удаление пробелов и приведение к одному регистру обычно идемпотентны; повторное добавление строк — нет. Для серьёзной задачи важно также документировать все исключения и сравнивать число входных и выходных строк.

**Проверьте себя:** зачем сохранять отбракованные строки? Как сформулировать правило корректной цены? Почему повторяемость обработки важнее «первый раз сработало»?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m3_16 · audit table

Получите dict с ключами `rows` (строк), `missing_cells` (всего NA), `duplicate_items` (число повторов item после первого, как в duplicated).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_16: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_16 = expected_for("m3_16", df=df)


In [ ]:
# m3_16: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'rows': 6, 'missing_cells': 4, 'duplicate_items': 1}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_16", result, _expected_m3_16)

### m3_17 · valid amount rows

Отберите строки, у которых `price` и `quantity` преобразуются в числа, `price>=0`, `quantity>=0`. Исходный индекс сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_17: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_17 = expected_for("m3_17", df=df)


In [ ]:
# m3_17: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
     item   price quantity category          email
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM
10   None       0        7     None               
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_17", result, _expected_m3_17)

### m3_18 · clean idempotent

Получите копию с нормализованными `item` (крайние пробелы убраны, верхний регистр) и `category` (крайние пробелы убраны, нижний регистр). Должна быть идемпотентность.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_18: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_18 = expected_for("m3_18", df=df)


In [ ]:
# m3_18: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
    item   price quantity category          email
11   A-1  128.75        2     food  A@EXAMPLE.COM
2    B-2                3     food        b@x.ru 
3    A-1  120.50        2     food  A@EXAMPLE.COM
8    C-3  ошибка     None      toy            bad
10  None       0        7     None               
15   D-4   -4.00        1      toy           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_18", result, _expected_m3_18)

### m3_19 · split valid invalid

Получите кортеж `(valid, invalid)`: валидны строки с числовой неотрицательной ценой и непустым (после strip) item. Ни одна строка не должна потеряться.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_19: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_19 = expected_for("m3_19", df=df)


In [ ]:
# m3_19: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Кортеж/список из 2 объектов: DataFrame (2, 5), DataFrame (4, 5).

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_19", result, _expected_m3_19)

### m3_20 · report rejections

Получите dict: `bad_price` — число строк с ошибочной или отрицательной price; `missing_item` — число пустых или отсутствующих item. Счётчики независимы.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m3_20: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m3_20 = expected_for("m3_20", df=df)


In [ ]:
# m3_20: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'bad_price': 3, 'missing_item': 1}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m3_20", result, _expected_m3_20)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()